# 02 — Portal Traversal and Discovery

Demonstrates SPARQL-driven portal discovery, CONSTRUCT-based traversal,
multi-hop path finding, membrane validation at each hop, and PROV-O
provenance recording.

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

# ══════════════════════════════════════════════════════════════
# Build three holons: Municipal → GeoNames → Wikidata
# Portals: muni→geo, geo→wiki (no direct muni→wiki)
# ══════════════════════════════════════════════════════════════

# ── Municipal source ──
ds.add_holon("urn:holon:municipal", "Municipal Data")
ds.add_interior("urn:holon:municipal", """
    @prefix muni: <urn:municipal:> .
    <urn:city:vancouver> a muni:CityRecord ;
        muni:officialName "City of Vancouver" ;
        muni:censusPop 675218 ;
        muni:lat 49.2827 ;
        muni:lon -123.1207 .
    <urn:city:victoria> a muni:CityRecord ;
        muni:officialName "City of Victoria" ;
        muni:censusPop 91867 ;
        muni:lat 48.4284 ;
        muni:lon -123.3656 .
""")

# ── GeoNames target/intermediate ──
ds.add_holon("urn:holon:geonames", "GeoNames Consumer")
ds.add_boundary("urn:holon:geonames", """
    @prefix geo: <urn:geonames:> .
    <urn:shapes:FeatureShape> a sh:NodeShape ;
        sh:targetClass geo:Feature ;
        sh:property [
            sh:path rdfs:label ;
            sh:minCount 1 ;
            sh:datatype xsd:string ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path geo:population ;
            sh:minCount 1 ;
            sh:datatype xsd:integer ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path geo:latitude ;
            sh:minCount 1 ;
            sh:datatype xsd:decimal ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path geo:longitude ;
            sh:minCount 1 ;
            sh:datatype xsd:decimal ;
            sh:severity sh:Violation
        ] .
""")

# ── Wikidata target ──
ds.add_holon("urn:holon:wikidata", "Wikidata Consumer")
ds.add_boundary("urn:holon:wikidata", """
    @prefix wd: <urn:wikidata:> .
    <urn:shapes:WikiEntityShape> a sh:NodeShape ;
        sh:targetClass wd:Entity ;
        sh:property [
            sh:path rdfs:label ;
            sh:minCount 1 ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path wd:population ;
            sh:datatype xsd:integer ;
            sh:severity sh:Warning
        ] .
""")

## Register portals

Portal definitions are RDF in boundary graphs — discoverable via SPARQL.

> Note: have to add something akin to the following in order to support getting the data (for the construct) from within the holons interior (i.e. any interior graph)
```
PREFIX cgn: <urn:holonic:ontology:> 
SELECT * WHERE { 
    graph <urn:holarchy:registry> {
        <urn:holon:name> cgn:hasInterior|cgn:hasContext|cgn:hasProjection ?o .
    }
}
```

In [ ]:
# Portal 1: Municipal → GeoNames
ds.add_portal(
    "urn:portal:muni-to-geo",
    "urn:holon:municipal",
    "urn:holon:geonames",
    label="Municipal → GeoNames",
    construct_query="""
        PREFIX muni: <urn:municipal:>
        PREFIX geo:  <urn:geonames:>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        CONSTRUCT {
            ?s a geo:Feature ;
                rdfs:label ?name ;
                geo:population ?pop ;
                geo:latitude ?lat ;
                geo:longitude ?lon .
        }
        WHERE {
            GRAPH <urn:holon:municipal/interior> {
                ?s a muni:CityRecord ;
                    muni:officialName ?name ;
                    muni:censusPop ?pop ;
                    muni:lat ?lat ;
                    muni:lon ?lon .
            }
        }
    """,
)

# Portal 2: GeoNames → Wikidata
ds.add_portal(
    "urn:portal:geo-to-wiki",
    "urn:holon:geonames",
    "urn:holon:wikidata",
    label="GeoNames → Wikidata",
    construct_query="""
        PREFIX geo:  <urn:geonames:>
        PREFIX wd:   <urn:wikidata:>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        CONSTRUCT {
            ?s a wd:Entity ;
                rdfs:label ?name ;
                wd:population ?pop .
        }
        WHERE {
            <urn:holon:geonames> cga:hasLayer ?g .
            
            graph ?g {
                GRAPH <urn:holon:geonames/context> {
                    ?s a geo:Feature ;
                        rdfs:label ?name .
                    OPTIONAL { ?s geo:population ?pop }
                }
            }
        }
    """,
)

print(ds.summary())

## Portal discovery via SPARQL

No Python iteration — the dataset is queried.

In [ ]:
print("Portals FROM municipal:")
for p in ds.find_portals_from("urn:holon:municipal"):
    print(f"  {p}")

print("\nPortals TO wikidata:")
for p in ds.find_portals_to("urn:holon:wikidata"):
    print(f"  {p}")

print("\nDirect portal municipal → wikidata:")
p = ds.find_portal("urn:holon:municipal", "urn:holon:wikidata")
print(f"  {p}")  # None — no direct route

## Multi-hop path finding

BFS over the SPARQL-discovered portal graph.

In [ ]:
path = ds.find_path("urn:holon:municipal", "urn:holon:wikidata")
if path:
    print(f"Path found ({len(path)} hops):")
    for i, p in enumerate(path):
        print(f"  Hop {i+1}: {p}")
else:
    print("No path found")

## Traverse hop-by-hop with validation at each step

In [ ]:
for i, portal_info in enumerate(path):
    print(f"\n{'='*60}")
    print(f"  Hop {i+1}: {portal_info.label}")
    print(f"{'='*60}")

    # Traverse this portal
    projected = ds.traverse_portal(
        portal_info.iri,
        inject_into=f"{portal_info.target_iri}/interior",
    )
    print(f"  Projected {len(projected)} triples")

    # Validate target membrane
    result = ds.validate_membrane(portal_info.target_iri)
    print(f"  {result.summary()}")

    # Record provenance
    ds.record_traversal(
        portal_iri=portal_info.iri,
        source_iri=portal_info.source_iri,
        target_iri=portal_info.target_iri,
        agent_iri="urn:agent:example-pipeline",
    )

## Inspect provenance trail

In [ ]:
print("\nWikidata context graph (provenance):")
g = ds.backend.get_graph("urn:holon:wikidata/context")
for s, p, o in sorted(g):
    s_short = str(s).rsplit(":", 1)[-1][:35]
    p_short = str(p).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
    o_short = str(o)[:50]
    print(f"  {s_short:37s} {p_short:30s} {o_short}")

### Provenance Visualization

In [ ]:
from holonic.viz import ProvenanceViz

pv = ProvenanceViz(ds)
pv.show()